# Next Priority Runs — Exp43 / VS r32

Outputs are Drive-only. This notebook does not update `PROGRESS.md` and does not push result CSVs to GitHub.

Preset order:
- `p0_exp43_s24`: Exp43 C0/C1 on S24, VS init r=32
- `p1_exp43_s01`: Exp43 C0/C1 on S01, VS init r=32
- `p2_vs_s28`: S28 r=32 SD LoRA VS generation
- `p3_vs_s02`: S02 r=32 SD LoRA VS generation


In [2]:
# [1] GPU check
import torch
assert torch.cuda.is_available(), 'GPU 없음: Runtime > Change runtime type > GPU'
print('torch', torch.__version__)
print('GPU  ', torch.cuda.get_device_name(0))


torch 2.11.0+cu128
GPU   NVIDIA L4


In [3]:
# [2] Mount Drive and clone/pull repo
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

if os.path.exists(WORKDIR) and os.path.exists(os.path.join(WORKDIR, '.git')):
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['rm', '-rf', WORKDIR], check=True)
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(['git', 'log', '--oneline', '-5'], check=False)


Mounted at /content/drive


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [ ]:
# [3] Select preset and launch background run
# Change PRESET one at a time. Do not run multiple presets simultaneously on one GPU.
PRESET = 'p1_exp43_s01' # 'p0_exp43_s24'  # p0_exp43_s24 | p1_exp43_s01 | p2_vs_s28 | p3_vs_s02

# Exp43 default uses VI/class=60 => 432 train trials. Set FULL_VI=True for all VI trials.
FULL_VI = False

cmd = f"python -u launch_next_priority_runs.py --preset {PRESET}"
if FULL_VI:
    cmd += ' --full_vi'

print(cmd)
!cd /content/vsvi_project && {cmd}


In [ ]:
# [4] Monitor active process and GPU
!pgrep -af 'train_exp43_vi_lora.py|train_vs_re_lora_gen.py' || true
!nvidia-smi
!ls -lt /content/drive/MyDrive/vsvi_data/logs | head -20


In [ ]:
# [5] Tail latest log for the selected preset
import glob, os
patterns = {
    'p0_exp43_s24': '/content/drive/MyDrive/vsvi_data/logs/exp43_s24_c0c1_r32_tok16_*.log',
    'p1_exp43_s01': '/content/drive/MyDrive/vsvi_data/logs/exp43_s01_c0c1_r32_tok16_*.log',
    'p2_vs_s28': '/content/drive/MyDrive/vsvi_data/logs/vs_s28_lora_r32_tok16_*.log',
    'p3_vs_s02': '/content/drive/MyDrive/vsvi_data/logs/vs_s02_lora_r32_tok16_*.log',
}
logs = sorted(glob.glob(patterns[PRESET]), key=os.path.getmtime, reverse=True)
assert logs, f'No log found for {PRESET}'
LOG = logs[0]
print('LOG=', LOG)
!tail -n 120 "{LOG}"


In [ ]:
# [6] Find Drive-only result CSVs
!echo '=== Exp43 VI results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora -name 'results_exp43_vi_lora.csv' -ls 2>/dev/null | tail -30
!echo '=== VS results ==='
!find /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen -name 'results_lora_gen.csv' -ls 2>/dev/null | tail -30


In [10]:
import os, shutil
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
from google.colab import drive
drive.mount('/content/drive')

OSError: [Errno 125] Operation canceled: '/content/drive/.Encrypted/.shortcut-targets-by-id'

In [9]:
%%bash
ls /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_01_1.npz && echo "✅ Drive OK" || echo "❌ Drive 문제"

/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_01_1.npz
✅ Drive OK


In [ ]:
%%bash
# 환경 복구
cd /content
git clone https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git vsvi_project 2>/dev/null
cd /content/vsvi_project && git pull
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re
ln -sfn /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino /content/vsvi_project/checkpoints_vsre_dino
pip uninstall -y torchao 2>/dev/null

ls /content/vsvi_project/preproc_vi_re/preproc_subj_02_1.npz && echo "✅ Data OK" || exit 1

# S02 Exp43
python -u train_exp43_vi_lora.py \
  --subject_ids 2 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260707_130522_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260707_133400_lora_r32_ep100/subj02_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 135 \
  --eval_n_samples 54 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s02_c0c1_r32_tok16_full_$(date +%Y%m%d_%H%M%S).log

In [ ]:
%%bash
cd /content/vsvi_project
rm -rf /content/vsvi_project/preproc_vs_re
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_dino.py \
  --subject_ids 28 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s28_$(date +%Y%m%d_%H%M%S).log

In [ ]:
%%bash
cd /content/vsvi_project

python -u train_vs_re_dino.py \
  --subject_ids 29 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s29_$(date +%Y%m%d_%H%M%S).log

In [ ]:
import os, shutil
# 잔여 제거
try:
    if os.path.exists('/content/drive'):
        shutil.rmtree('/content/drive')
except:
    pass

from google.colab import drive
drive.mount('/content/drive')
print("Drive:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re'))

In [6]:
import os
# 마운트 상태 확인
print("MyDrive:", os.path.exists('/content/drive/MyDrive'))
print("내용:", os.listdir('/content/drive/MyDrive')[:10] if os.path.exists('/content/drive/MyDrive') else "없음")

MyDrive: True
내용: ['Quince · SlidesCarnival의 사본.gslides', 'Hyperscanning_', 'DSI 24.pptx', 'EEG&연구설명 발표자료.pptx', 'Print', 'vsvi_data', 'MI_loso_project', 'project.tar.gz']


In [7]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
print("vsvi_data:", os.path.exists('/content/drive/MyDrive/vsvi_data'))

Mounted at /content/drive
vsvi_data: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
print("vsvi_data:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vi_re'))

In [ ]:
import os
if not os.path.exists('/content/drive/MyDrive/vsvi_data'):
    from google.colab import drive
    drive.mount('/content/drive')
print("Drive:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
print("vsvi_data:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re'))
print("S09 파일:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_09_1.npz'))

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("S09 vs:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_09_1.npz'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
S09 vs: True


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
print("vi_re:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_09_1.npz'))
print("vs_re:", os.path.exists('/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_09_1.npz'))

Mounted at /content/drive
vi_re: True
vs_re: True


In [12]:
%%bash
cd /content/vsvi_project

# 실제 디렉터리 제거 (rm -rf)
rm -rf /content/vsvi_project/preproc_vs_re
rm -rf /content/vsvi_project/preproc_vi_re

# symlink 생성
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

# 확인
ls -la preproc_vs_re/preproc_subj_09_1.npz && echo "✅ OK" || echo "❌ 실패"

-rw------- 1 root root 16132799 Jul  9 18:05 preproc_vs_re/preproc_subj_09_1.npz
✅ OK


In [13]:
%%bash
cd /content/vsvi_project 2>/dev/null || (cd /content && git clone https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git vsvi_project && cd vsvi_project)
cd /content/vsvi_project
rm -f preproc_vs_re
ln -sfn /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

ls preproc_vs_re/preproc_subj_09_1.npz && echo "✅ OK" || exit 1

python -u train_vs_re_dino.py \
  --subject_ids 9 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s09_$(date +%Y%m%d_%H%M%S).log

preproc_vs_re/preproc_subj_09_1.npz
✅ OK
[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [9]
[INFO] Session counts: {9: 4}
[INFO] Loading DINO: dinov2_vits14
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vi

In [14]:
%%bash
cd /content/vsvi_project

# S09 데이터에 NaN이나 이상값이 있는지 확인
python3 -c "
import numpy as np
import glob

for sid in [9, 28, 29]:
    files = sorted(glob.glob(f'/content/vsvi_project/preproc_vs_re/preproc_subj_{sid:02d}_*.npz'))
    print(f'=== S{sid:02d} ({len(files)} sessions) ===')
    for f in files[:2]:
        d = np.load(f)
        keys = list(d.keys())
        X = d[keys[0]]
        print(f'  {f.split(\"/\")[-1]}: shape={X.shape}, NaN={np.isnan(X).any()}, min={X.min():.3f}, max={X.max():.3f}, mean={X.mean():.3f}')
"

=== S09 (4 sessions) ===
  preproc_subj_09_1.npz: shape=(135, 32, 2048), NaN=False, min=-242.250, max=397.500, mean=-0.609
  preproc_subj_09_2.npz: shape=(135, 32, 2048), NaN=False, min=-545.000, max=596.500, mean=-0.358
=== S28 (6 sessions) ===
  preproc_subj_28_1.npz: shape=(135, 32, 2048), NaN=False, min=-inf, max=inf, mean=nan
  preproc_subj_28_2.npz: shape=(135, 32, 2048), NaN=False, min=-158.625, max=285.750, mean=-0.360
=== S29 (5 sessions) ===
  preproc_subj_29_1.npz: shape=(135, 32, 2048), NaN=False, min=-219.750, max=360.000, mean=-0.312
  preproc_subj_29_2.npz: shape=(135, 32, 2048), NaN=False, min=-276.250, max=380.250, mean=-0.338


/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:127: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


In [15]:
%%bash
cd /content/vsvi_project
python3 -c "
import numpy as np, glob
files = sorted(glob.glob('/content/vsvi_project/preproc_vs_re/preproc_subj_28_*.npz'))
for f in files:
    d = np.load(f)
    X = d[list(d.keys())[0]]
    n_inf = np.isinf(X).sum()
    print(f'{f.split(\"/\")[-1]}: inf 개수={n_inf}, shape={X.shape}')
"

preproc_subj_28_1.npz: inf 개수=44767, shape=(135, 32, 2048)
preproc_subj_28_2.npz: inf 개수=0, shape=(135, 32, 2048)
preproc_subj_28_3.npz: inf 개수=0, shape=(135, 32, 2048)
preproc_subj_28_4.npz: inf 개수=0, shape=(135, 32, 2048)
preproc_subj_28_5.npz: inf 개수=0, shape=(108, 32, 2048)
preproc_subj_28_6.npz: inf 개수=0, shape=(135, 32, 2048)


In [16]:
%%bash
cd /content/vsvi_project

python -u train_vs_re_dino.py \
  --subject_ids 9 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  --lr 1e-4 \
  --temperature 0.07 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s09_retry_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [9]
[INFO] Session counts: {9: 4}
[INFO] Loading DINO: dinov2_vits14
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
  proto_dino: torch.Size([9, 384])

  Subject 09  (sessions=4)
  [dataset] S09 sess1: npz  shape=(135, 32, 2048)
  [dataset] S09 sess2: npz  shape=(135, 32, 2048)
  [dataset] S09 sess3: npz  shape=(135, 32, 2048)
  [dataset] S09 sess4: npz  shape=(108, 32, 2048)
  [dataset] S09  trials=5

In [17]:
%%bash
# S28 sess1을 백업 (로더가 안 읽도록 확장자 변경)
mv /content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_1.npz \
   /content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_1.npz.bad

# 확인
ls /content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_*.npz

/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_2.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_3.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_4.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_5.npz
/content/drive/MyDrive/vsvi_data/preproc_vs_re/preproc_subj_28_6.npz


In [18]:
%%bash
cd /content/vsvi_project
rm -rf preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_dino.py \
  --subject_ids 28 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  --lr 1e-4 \
  --temperature 0.07 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s28_retry_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [28]
[INFO] Session counts: {28: 5}
[INFO] Loading DINO: dinov2_vits14
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
  proto_dino: torch.Size([9, 384])

  Subject 28  (sessions=5)
  [dataset] S28 sess2: npz  shape=(135, 32, 2048)
  [dataset] S28 sess3: npz  shape=(135, 32, 2048)
  [dataset] S28 sess4: npz  shape=(135, 32, 2048)
  [dataset] S28 sess5: npz  shape=(108, 32, 2048)
  [dataset] S28 sess6: 

In [19]:
%%bash
cd /content/vsvi_project

python -u train_vs_re_dino.py \
  --subject_ids 29 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino \
  --loss_type supcon \
  --encoder_type v2 \
  --epochs 200 \
  --batch_size 4 \
  --lr 1e-4 \
  --temperature 0.07 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/supcon_s29_retry_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  n_ch=32  max_sessions=None  loss_type=supcon
[INFO] Subjects (1): [29]
[INFO] Session counts: {29: 5}
[INFO] Loading DINO: dinov2_vits14
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
  proto_dino: torch.Size([9, 384])

  Subject 29  (sessions=5)
  [dataset] S29 sess1: npz  shape=(135, 32, 2048)
  [dataset] S29 sess2: npz  shape=(135, 32, 2048)
  [dataset] S29 sess3: npz  shape=(135, 32, 2048)
  [dataset] S29 sess4: npz  shape=(135, 32, 2048)
  [dataset] S29 sess5: 

In [21]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project

python -u train_vs_re_lora_gen.py \
  --subject_ids 28 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_190207_ch32_merged_ep200_supcon \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/vs_lora_s28_r32_$(date +%Y%m%d_%H%M%S).log

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
[INFO] Device: cuda  SD1.5 LoRA generator
[INFO] Loading DINO...
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] Loading SD VAE...
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migratin

In [22]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project
rm -rf preproc_vi_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

python -u train_exp43_vi_lora.py \
  --subject_ids 28 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_190207_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 90 \
  --eval_n_samples 54 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s28_c0c1_r32_tok16_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  Exp43 VI LoRA
[INFO] Loading DINO...
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] Loading SD VAE...
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_

In [3]:
%%bash
cd /content/vsvi_project
python3 -c "
import numpy as np, glob
files = sorted(glob.glob('/content/vsvi_project/preproc_vi_re/preproc_subj_28_*.npz'))
for f in files:
    d = np.load(f)
    X = d[list(d.keys())[0]]
    n_inf = np.isinf(X).sum()
    n_nan = np.isnan(X).sum()
    print(f'{f.split(\"/\")[-1]}: inf={n_inf}, nan={n_nan}, shape={X.shape}')
"

bash: line 1: cd: /content/vsvi_project: No such file or directory


In [24]:
%%bash
cd /content/vsvi_project
python3 -c "
import numpy as np, glob
for sid in [28, 29, 9]:
    files = sorted(glob.glob(f'/content/vsvi_project/preproc_vi_re/preproc_subj_{sid:02d}_*.npz'))
    print(f'=== S{sid:02d} VI ===')
    for f in files:
        d = np.load(f)
        X = d[list(d.keys())[0]]
        print(f'  {f.split(\"/\")[-1]}: inf={np.isinf(X).sum()}, nan={np.isnan(X).sum()}')
"

=== S28 VI ===
  preproc_subj_28_1.npz: inf=0, nan=0
  preproc_subj_28_2.npz: inf=0, nan=0
  preproc_subj_28_3.npz: inf=0, nan=0
  preproc_subj_28_4.npz: inf=0, nan=0
  preproc_subj_28_5.npz: inf=33673, nan=0
  preproc_subj_28_6.npz: inf=0, nan=0
=== S29 VI ===
  preproc_subj_29_1.npz: inf=0, nan=0
  preproc_subj_29_2.npz: inf=0, nan=0
  preproc_subj_29_3.npz: inf=0, nan=0
  preproc_subj_29_4.npz: inf=0, nan=0
  preproc_subj_29_5.npz: inf=0, nan=0
=== S09 VI ===
  preproc_subj_09_1.npz: inf=0, nan=0
  preproc_subj_09_2.npz: inf=0, nan=0
  preproc_subj_09_3.npz: inf=0, nan=0
  preproc_subj_09_4.npz: inf=0, nan=0


In [3]:
%%bash
# S28 VI sess5 제외
mv /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_5.npz \
   /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_5.npz.bad

# 확인
ls /content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_*.npz

/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_1.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_2.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_3.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_4.npz
/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_6.npz


mv: cannot stat '/content/drive/MyDrive/vsvi_data/preproc_vi_re/preproc_subj_28_5.npz': No such file or directory


In [4]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project
rm -rf preproc_vi_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

python -u train_exp43_vi_lora.py \
  --subject_ids 28 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_190207_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260709_191616_lora_r32_ep100/subj28_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 75 \
  --eval_n_samples 45 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s28_c0c1_r32_tok16_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  Exp43 VI LoRA
[INFO] Loading DINO...
[INFO] local DINO cache not found, downloading facebookresearch/dinov2...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:00<00:00, 228M

In [4]:
%%bash
pip uninstall -y torchao 2>/dev/null
cd /content/vsvi_project
rm -rf preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_lora_gen.py \
  --subject_ids 29 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_191010_ch32_merged_ep200_supcon \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/vs_lora_s29_r32_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  SD1.5 LoRA generator
[INFO] Loading DINO...
[INFO]  local cache not found, downloading from facebookresearch/dinov2...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:00<00:

In [6]:
%%bash
pip uninstall -y torchao 2>/dev/null

cd /content/vsvi_project
rm -rf preproc_vi_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vi_re /content/vsvi_project/preproc_vi_re

python -u train_exp43_vi_lora.py \
  --subject_ids 29 \
  --conditions c0,c1 \
  --data_root /content/vsvi_project/preproc_vi_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_191010_ch32_merged_ep200_supcon \
  --init_lora_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/20260710_003529_lora_r32_ep100/subj29_lora_best.pt \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_exp43_vi_lora \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --per_class_total 75 \
  --eval_n_samples 45 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/exp43_s29_c0c1_r32_tok16_$(date +%Y%m%d_%H%M%S).log

Process is terminated.


In [8]:
%%bash
pip uninstall -y torchao 2>/dev/null
cd /content/vsvi_project
rm -rf preproc_vs_re
ln -s /content/drive/MyDrive/vsvi_data/preproc_vs_re /content/vsvi_project/preproc_vs_re

python -u train_vs_re_lora_gen.py \
  --subject_ids 9 \
  --data_root /content/vsvi_project/preproc_vs_re \
  --img_root /content/vsvi_project/preproc_data_vi/images \
  --supcon_ckpt /content/drive/MyDrive/vsvi_data/checkpoints_vsre_dino/20260709_185832_ch32_merged_ep200_supcon \
  --ckpt_root /content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen \
  --lora_r 32 \
  --n_eeg_tokens 16 \
  --epochs 100 \
  --batch_size 2 \
  --fp16 \
  2>&1 | tee /content/drive/MyDrive/vsvi_data/logs/vs_lora_s09_r32_$(date +%Y%m%d_%H%M%S).log

[INFO] Device: cuda  SD1.5 LoRA generator
[INFO] Loading DINO...
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
[INFO] Loading SD VAE...
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/

In [40]:
%%bash
sed -n '164,185p' /content/vsvi_project/train_vs_re_lora_gen.py
echo "=== make_schedule 반환 ==="
grep -n "def make_schedule" /content/vsvi_project/train_vs_re_latent_gen.py
sed -n '/def make_schedule/,/return/p' /content/vsvi_project/train_vs_re_latent_gen.py | head -25

def sample_sd_ddim(unet, cond_proj, eeg_lat, acp, T, steps=50, device='cuda', guidance=1.0):
    """DDIM sampling with SD 1.5 UNet conditioned on EEG."""
    B = eeg_lat.size(0)
    cond_tokens = cond_proj(eeg_lat)  # (B, N_tok, 768)

    seq = list(range(0, T, T // steps))
    x = torch.randn(B, 4, 64, 64, device=device)

    for i in reversed(range(len(seq))):
        t_val = seq[i]
        t_tensor = torch.full((B,), t_val, dtype=torch.long, device=device)
        noise_pred = unet(x, t_tensor, encoder_hidden_states=cond_tokens).sample
        a  = acp[seq[i]]
        a_ = acp[seq[i-1]] if i > 0 else torch.tensor(1.0, device=device)
        x0_pred = (x - (1-a).sqrt() * noise_pred) / a.sqrt()
        x0_pred = x0_pred.clamp(-1, 1)
        x = a_.sqrt() * x0_pred + (1 - a_).sqrt() * noise_pred
    return x


@torch.no_grad()
def evaluate_lora(unet, cond_proj, eeg_enc, test_loader, vae, dino,
=== make_schedule 반환 ===
267:def make_schedule(T, device):
def make_schedule(T, device):
    

In [41]:
%%bash
grep -n "acp\|make_schedule" /content/vsvi_project/train_exp43_vi_lora.py | head -10

40:from train_vs_re_latent_gen import build_eeg_encoder, make_schedule
337:    acp,
423:            xt, noise = ddpm_q_sample(x0, t, acp)
481:        acp,
600:    _, _, acp = make_schedule(args.num_timesteps, device)
675:                acp,
